In [1]:
from jax import numpy as jnp
import jax
import numpy as np

In [2]:
def model(inputs, W, b):
    return jnp.matmul(inputs, W) + b

def mean_squared_error(targets, predictions):
    per_sample_losses = jnp.square(targets - predictions)
    return jnp.mean(per_sample_losses)

In [3]:
def compute_loss(state, inputs, targets):
    W, b = state
    predictions = model(inputs, W, b)
    loss = mean_squared_error(targets, predictions)
    return loss

In [4]:
grad_fn = jax.value_and_grad(compute_loss)

In [5]:
learning_rate = 0.1
@jax.jit
def training_step(inputs, targets, W, b):
    loss, grads = grad_fn((W, b), inputs, targets)
    grad_wrt_W, grad_wrt_b = grads
    W = W - grad_wrt_W * learning_rate
    b = b - grad_wrt_b * learning_rate
    return loss, W, b

In [6]:
# Synthetic Data
num_samples_per_class = 1000
negative_samples = np.random.multivariate_normal(mean=[0, 3], cov=[[1, 0.55], [0.55, 1]], size=num_samples_per_class)
positive_samples = np.random.multivariate_normal(mean=[3, 0], cov=[[1, 0.55], [0.55, 1]], size=num_samples_per_class)
inputs = np.vstack((negative_samples, positive_samples)).astype(np.float32)
targets = np.vstack((np.zeros((num_samples_per_class, 1), dtype="float32"), np.ones((num_samples_per_class, 1), dtype="float32")))


In [7]:
input_dim = 2
output_dim = 1
W = jax.numpy.array(np.random.uniform(size=(input_dim, output_dim)))
b = jax.numpy.array(np.zeros(shape=(output_dim,)))
state = (W, b)
for step in range(40):
    loss, W, b = training_step(inputs, targets, W, b)
    print(f"Loss at step {step}: {loss:.4f}")

Loss at step 0: 0.4247
Loss at step 1: 0.1385
Loss at step 2: 0.0990
Loss at step 3: 0.0877
Loss at step 4: 0.0813
Loss at step 5: 0.0760
Loss at step 6: 0.0713
Loss at step 7: 0.0670
Loss at step 8: 0.0631
Loss at step 9: 0.0595
Loss at step 10: 0.0563
Loss at step 11: 0.0533
Loss at step 12: 0.0506
Loss at step 13: 0.0482
Loss at step 14: 0.0460
Loss at step 15: 0.0439
Loss at step 16: 0.0421
Loss at step 17: 0.0404
Loss at step 18: 0.0388
Loss at step 19: 0.0374
Loss at step 20: 0.0362
Loss at step 21: 0.0350
Loss at step 22: 0.0340
Loss at step 23: 0.0330
Loss at step 24: 0.0321
Loss at step 25: 0.0313
Loss at step 26: 0.0306
Loss at step 27: 0.0299
Loss at step 28: 0.0293
Loss at step 29: 0.0288
Loss at step 30: 0.0283
Loss at step 31: 0.0278
Loss at step 32: 0.0274
Loss at step 33: 0.0270
Loss at step 34: 0.0267
Loss at step 35: 0.0263
Loss at step 36: 0.0261
Loss at step 37: 0.0258
Loss at step 38: 0.0256
Loss at step 39: 0.0253


In [8]:
predictions = model(inputs, W, b)
predicted_labels = predictions[:, 0] > 0.5
matches = (predicted_labels == targets.flatten())
accuracy = jnp.mean(matches)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 99.95%
